# Jour 3 · Projet final — assistant helpdesk observable


## Mission

Assembler les couches étudiées sans les confondre : documents persistés, tool RAG, règle SLA, agent avec mémoire courte et interface Streamlit.

Le projet est réussi lorsque nous pouvons expliquer l'entrée et la sortie de chaque couche, puis localiser un échec.


![Architecture finale du système](../../assets/jour_01/13_architecture_complete.png)

*L'agent choisit entre des tools testables ; Streamlit reste une interface au-dessus du cœur Python.*


## Critères de réussite

1. Le tool RAG renvoie des passages avec leurs métadonnées.
2. Le tool SLA applique une règle Python déterministe.
3. L'agent choisit l'outil adapté à la question.
4. Les sources affichées proviennent des observations du tool.
5. Un même `thread_id` conserve le fil de la conversation.
6. Les itérations de l'agent sont limitées.
7. L'interface et le cœur métier sont séparés.
8. Les tests vérifient outil, source et réponse séparément.


In [ ]:
import sys
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Les imports Python ne suivent pas automatiquement les chemins de fichiers.
# Cette ligne permet notamment d'importer le package local `apps` depuis un
# notebook lancé dans jour_03 ou corrections/jour_03.
root_text = str(ROOT)
if root_text not in sys.path:
    sys.path.insert(0, root_text)

print("Racine du cours :", ROOT)


## Étape A — construire le cœur

Réutilisez les fonctions de `apps/helpdesk_core.py`. Ne dupliquez pas le code de l'application dans le notebook.


In [ ]:
from apps.helpdesk_core import ask_agent, build_agent


agent_projet = build_agent()
print("Agent du projet construit.")


## Étape B — définir les attentes

Préparez au moins une question documentaire, une question SLA, une question de sécurité et une question ne nécessitant aucun outil.


In [ ]:
cas_finaux = [
    {
        "question": "Quel portail dois-je saisir dans GlobalProtect ?",
        "outil_attendu": "chercher_procedure",
        "source_attendue": "procedure_vpn.md",
    },
    {
        "question": "Quel SLA pour un ticket CRITIQUE avec 400 personnes bloquées ?",
        "outil_attendu": "calculer_sla",
        "source_attendue": None,
    },
    {
        "question": "Puis-je donner mon code MFA au support ?",
        "outil_attendu": "chercher_procedure",
        "source_attendue": "politique_securite.md",
    },
]
print("Cas finaux :", len(cas_finaux))


## Étape C — observer chaque couche

Une réponse agréable ne suffit pas. Vérifiez le tool demandé, ses arguments, la présence de la source puis la réponse finale.


![Évaluation couche par couche](../../assets/jour_01/13_evaluation_finale.png)

*Un résultat bout-en-bout ne permet pas toujours d'identifier la première couche incorrecte.*


In [ ]:
def evaluer_agent(agent, cas):
    lignes = []
    for index, item in enumerate(cas):
        resultat = ask_agent(agent, item["question"], thread_id=f"evaluation-{index}")
        outils = [appel["name"] for appel in resultat["tool_calls"]]
        bon_outil = item["outil_attendu"] in outils
        bonne_source = (
            item["source_attendue"] is None
            or item["source_attendue"] in resultat["sources"]
        )
        lignes.append({
            "question": item["question"],
            "outils": outils,
            "sources": resultat["sources"],
            "bon_outil": bon_outil,
            "bonne_source": bonne_source,
            "reponse": resultat["answer"],
        })
    return lignes


resultats_finaux = evaluer_agent(agent_projet, cas_finaux)
for ligne in resultats_finaux:
    print("\nQUESTION :", ligne["question"])
    print("OUTILS   :", ligne["outils"], "OK" if ligne["bon_outil"] else "À ANALYSER")
    print("SOURCES  :", ligne["sources"], "OK" if ligne["bonne_source"] else "À ANALYSER")
    print("RÉPONSE  :", ligne["reponse"])


## Étape D — tester une conversation

Réutilisez le même `thread_id` pour le suivi. Changez-le ensuite et vérifiez que la nouvelle conversation ne récupère pas l'ancien contexte.


In [ ]:
premier = ask_agent(
    agent_projet,
    "Mon problème concerne GlobalProtect.",
    thread_id="conversation-projet",
)
suivi = ask_agent(
    agent_projet,
    "Et quel portail dois-je saisir ?",
    thread_id="conversation-projet",
)

print("Premier tour :", premier["answer"])
print("Suivi        :", suivi["answer"])
print("Sources      :", suivi["sources"])


## Avant une vraie mise en production

Ce projet reste une démonstration locale. Une mise en production demande notamment : authentification, droits par utilisateur, stockage durable des conversations, secrets protégés, logs sans données sensibles, supervision, sauvegarde, tests de charge et validation humaine des actions importantes.

« Local » signifie que le modèle tourne sur cette machine ; cela ne garantit pas automatiquement la conformité ou la sécurité de toute l'application.


## Bilan des trois journées

```text
documents → chunks → embeddings → Chroma
                          ↓
conversation → question autonome → tool RAG
                          ↓
           agent → observations → réponse
                          ↓
            Streamlit → utilisateur
```

La règle de conception reste la même : une couche, une responsabilité, un résultat observable et un test adapté.
